# 1. Inicialização

In [27]:
pip install xgboost

In [28]:
!hf auth login

User is already logged in.


In [29]:
pip install tabpfn

In [30]:
import pandas as pd
from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from huggingface_hub import hf_hub_download
from sklearn.metrics import confusion_matrix
from sklearn.metrics import matthews_corrcoef

In [31]:
df = pd.read_csv("https://github.com/Nertonm/tabpfn-xgboost-fairness-comparison/raw/refs/heads/main/data/prefeito_ceara.csv", encoding='latin1', sep=';')
df2 = pd.read_csv("https://github.com/Nertonm/tabpfn-xgboost-fairness-comparison/raw/refs/heads/main/data/prefeito_ceara2.csv", encoding='utf-8', sep=';')
df2 = df2.drop("Região", axis=1)
df3 = pd.read_csv("https://github.com/Nertonm/tabpfn-xgboost-fairness-comparison/raw/refs/heads/main/data/prefeito_ceara3.csv", encoding='utf-8', sep=';')
df3 = df3.drop("Região", axis=1)

In [32]:
df = pd.concat([df, df2, df3], ignore_index=True)

In [33]:
display(df)

,Código município,Cor/raça,Estado civil,Faixa etária,Gênero,Grau de instrução,Município,Nome candidato,Número candidato,Ocupação,Partido,Situação totalização,Votos válidos,Votos nominais,Data de carga
0,14630,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,MAURITI,JOÃO PAULO FURTADO,13,Prefeito,PT,Eleito,28644,20593,2026-02-28 19:30:47
1,13358,Parda,Divorciado(a),45 a 49 anos,Masculino,Ensino Médio completo,BAIXIO,LUCIO ALVES BARROSO,10,Empresário,REPUBLICANOS,Eleito,4916,2542,2026-02-28 19:30:47
2,13897,Branca,Casado(a),55 a 59 anos,Masculino,Superior completo,FORTALEZA,EVANDRO SA BARRETO LEITAO,13,Deputado,PT,Eleito,1421428,716133,2026-02-28 19:30:47
3,15997,Parda,Divorciado(a),60 a 64 anos,Feminino,Superior completo,PARAIPABA,JOANA D' ARC BATISTA CARVALHO,55,Aposentado (Exceto Servidor Público),PSD,Não Eleito,20496,0,2026-02-28 19:30:47
4,14753,Branca,Casado(a),35 a 39 anos,Masculino,Superior completo,MORADA NOVA,MARCO ANTONIO DE ARAUJO BICA JUNIOR,13,Advogado,PT,Não Eleito,48016,23206,2026-02-28 19:30:47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1536,15857,Parda,Casado(a),50 a 54 anos,Masculino,Superior completo,MARACANAÚ,JOSE FIRMO CAMURÇA NETO,22,Prefeito,PR,Eleito,112784,81315,2026-01-19 14:01:28
1537,14915,Parda,Casado(a),35 a 39 anos,Masculino,Superior completo,ORÓS,SIMAO PEDRO ALVES PEQUENO,55,Prefeito,PSD,Eleito,14084,8055,2026-01-19 14:01:28
1538,13080,Parda,Casado(a),75 a 79 anos,Masculino,Superior completo,TURURU,RAIMUNDO SERPA BARROSO,15,Vereador,PMDB,Não Eleito,11534,4998,2026-01-19 14:01:28
1539,15350,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,RERIUTABA,CLERTON ASSIS FURTADO,13,Empresário,PT,Não Eleito,11406,2746,2026-01-19 14:01:28


# 2. Tratamento dos dados

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1541 entries, 0 to 1540
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Código município      1541 non-null   int64 
 1   Cor/raça              1541 non-null   object
 2   Estado civil          1541 non-null   object
 3   Faixa etária          1541 non-null   object
 4   Gênero                1541 non-null   object
 5   Grau de instrução     1541 non-null   object
 6   Município             1541 non-null   object
 7   Nome candidato        1541 non-null   object
 8   Número candidato      1541 non-null   int64 
 9   Ocupação              1541 non-null   object
 10  Partido               1541 non-null   object
 11  Situação totalização  1541 non-null   object
 12  Votos válidos         1541 non-null   int64 
 13  Votos nominais        1541 non-null   int64 
 14  Data de carga         1541 non-null   object
dtypes: int64(4), object(11)
memory usage: 

In [35]:
df = df[df["Situação totalização"] != "Segundo turno"]

In [36]:
df = df.drop(["Nome candidato", "Data de carga", "Município"], axis=1)

In [37]:
df["Código cor"] = df["Cor/raça"].astype("category").cat.codes
df["Código civil"] = df["Estado civil"].astype("category").cat.codes
df["Código etária"] = df["Faixa etária"].astype("category").cat.codes
df["Código gênero"] = df["Gênero"].astype("category").cat.codes
df["Código instrução"] = df["Grau de instrução"].astype("category").cat.codes
df["Código ocupação"] = df["Ocupação"].astype("category").cat.codes
df["Código partido"] = df["Partido"].astype("category").cat.codes
df["Eleito"] = (df["Situação totalização"] == "Eleito").astype(int)

In [38]:
display(df)

,Código município,Cor/raça,Estado civil,Faixa etária,Gênero,Grau de instrução,Número candidato,Ocupação,Partido,Situação totalização,Votos válidos,Votos nominais,Código cor,Código civil,Código etária,Código gênero,Código instrução,Código ocupação,Código partido,Eleito
0,14630,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,13,Prefeito,PT,Eleito,28644,20593,4,0,4,1,5,56,38,1
1,13358,Parda,Divorciado(a),45 a 49 anos,Masculino,Ensino Médio completo,10,Empresário,REPUBLICANOS,Eleito,4916,2542,4,1,5,1,2,26,45,1
2,13897,Branca,Casado(a),55 a 59 anos,Masculino,Superior completo,13,Deputado,PT,Eleito,1421428,716133,1,0,7,1,5,21,38,1
3,15997,Parda,Divorciado(a),60 a 64 anos,Feminino,Superior completo,55,Aposentado (Exceto Servidor Público),PSD,Não Eleito,20496,0,4,1,8,0,5,6,32,0
4,14753,Branca,Casado(a),35 a 39 anos,Masculino,Superior completo,13,Advogado,PT,Não Eleito,48016,23206,1,0,3,1,5,1,38,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1536,15857,Parda,Casado(a),50 a 54 anos,Masculino,Superior completo,22,Prefeito,PR,Eleito,112784,81315,4,0,6,1,5,56,24,1
1537,14915,Parda,Casado(a),35 a 39 anos,Masculino,Superior completo,55,Prefeito,PSD,Eleito,14084,8055,4,0,3,1,5,56,32,1
1538,13080,Parda,Casado(a),75 a 79 anos,Masculino,Superior completo,15,Vereador,PMDB,Não Eleito,11534,4998,4,0,11,1,5,83,18,0
1539,15350,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,13,Empresário,PT,Não Eleito,11406,2746,4,0,4,1,5,26,38,0


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1529 entries, 0 to 1540
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Código município      1529 non-null   int64 
 1   Cor/raça              1529 non-null   object
 2   Estado civil          1529 non-null   object
 3   Faixa etária          1529 non-null   object
 4   Gênero                1529 non-null   object
 5   Grau de instrução     1529 non-null   object
 6   Número candidato      1529 non-null   int64 
 7   Ocupação              1529 non-null   object
 8   Partido               1529 non-null   object
 9   Situação totalização  1529 non-null   object
 10  Votos válidos         1529 non-null   int64 
 11  Votos nominais        1529 non-null   int64 
 12  Código cor            1529 non-null   int8  
 13  Código civil          1529 non-null   int8  
 14  Código etária         1529 non-null   int8  
 15  Código gênero         1529 non-null   int8 

In [40]:
df = df.drop(["Situação totalização", "Cor/raça", "Estado civil", "Faixa etária", "Gênero", "Grau de instrução", "Ocupação", "Partido", "Votos válidos", "Votos nominais"], axis=1)

In [41]:
display(df)

,Código município,Número candidato,Código cor,Código civil,Código etária,Código gênero,Código instrução,Código ocupação,Código partido,Eleito
0,14630,13,4,0,4,1,5,56,38,1
1,13358,10,4,1,5,1,2,26,45,1
2,13897,13,1,0,7,1,5,21,38,1
3,15997,55,4,1,8,0,5,6,32,0
4,14753,13,1,0,3,1,5,1,38,0
...,...,...,...,...,...,...,...,...,...,...
1536,15857,22,4,0,6,1,5,56,24,1
1537,14915,55,4,0,3,1,5,56,32,1
1538,13080,15,4,0,11,1,5,83,18,0
1539,15350,13,4,0,4,1,5,26,38,0


# 3. Treinamento e teste dos modelos

## 3.1. Divisão em teste e treino

In [42]:
X = df.drop("Eleito", axis=1)
y = df["Eleito"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 3.2. XGBoost

In [43]:
xgboost = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgboost.fit(X_train, y_train)
y_pred_xgboost = xgboost.predict(X_test)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:28:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [44]:
print(classification_report(y_test, y_pred_xgboost))

              precision    recall  f1-score   support

           0       0.71      0.68      0.69       193
           1       0.49      0.52      0.50       113

    accuracy                           0.62       306
   macro avg       0.60      0.60      0.60       306
weighted avg       0.63      0.62      0.62       306



## 3.3. TabPFN

In [47]:
tabpfn = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
tabpfn.fit(X_train, y_train)
y_pred_tabpfn = tabpfn.predict(X_test)

In [48]:
print(classification_report(y_test, y_pred_tabpfn))

              precision    recall  f1-score   support

           0       0.71      0.85      0.77       193
           1       0.61      0.41      0.49       113

    accuracy                           0.69       306
   macro avg       0.66      0.63      0.63       306
weighted avg       0.67      0.69      0.67       306



# 4. Comparação entre os modelos

## 4.1. Matriz de confusão

### 4.1.1. XGBoost

In [49]:
print(confusion_matrix(y_test, y_pred_xgboost))

[[131  62]
 [ 54  59]]


### 4.1.2. TabPFN

In [50]:
print(confusion_matrix(y_test, y_pred_tabpfn))

[[164  29]
 [ 67  46]]


## 4.2. Matthews Correlation Coeficient (MCC)

### 4.2.1. XGBoost

In [51]:
mcc_xgb = matthews_corrcoef(y_test, y_pred_xgboost)
print(mcc_xgb)

0.19827904915004743


### 4.2.2. TabPFN

In [52]:
mcc_pfn = matthews_corrcoef(y_test, y_pred_tabpfn)
print(mcc_pfn)

0.2881449912679822


# 5. Análise de viés

## 5.1. Taxa de vitória real entre os gêneros

In [53]:
real_rates = df.groupby('Código gênero')['Eleito'].mean()
print("Taxa real de vitória:")
print(real_rates)

Taxa real de vitória:
Código gênero
0    0.379447
1    0.365987
Name: Eleito, dtype: float64


## 5.2. Previsão por gênero

### 5.2.1. XGBoost

In [54]:
df_test_xg = X_test.copy()
df_test_xg['y_true'] = y_test
df_test_xg['y_pred'] = y_pred_xgboost

pred_rates = df_test_xg.groupby('Código gênero')['y_pred'].mean()

print("\nTaxa predita de vitória:")
print(pred_rates)


Taxa predita de vitória:
Código gênero
0    0.416667
1    0.391473
Name: y_pred, dtype: float64


### 5.2.2. TabPFN

In [55]:
df_test_pfn = X_test.copy()
df_test_pfn['y_true'] = y_test
df_test_pfn['y_pred'] = y_pred_tabpfn

pred_rates = df_test_pfn.groupby('Código gênero')['y_pred'].mean()

print("\nTaxa predita de vitória:")
print(pred_rates)


Taxa predita de vitória:
Código gênero
0    0.145833
1    0.263566
Name: y_pred, dtype: float64


## 5.3. Falsos negativos por gênero

In [56]:
def fnr_by_group(data, gender_value):
    subset = data[data['Código gênero'] == gender_value]
    tn, fp, fn, tp = confusion_matrix(
        subset['y_true'],
        subset['y_pred']
    ).ravel()
    return fn / (fn + tp)

### 5.3.1. XGBoost

In [57]:
fnr_men = fnr_by_group(df_test_xg, 1)
fnr_women = fnr_by_group(df_test_xg, 0)

print("FNR Homens:", fnr_men)
print("FNR Mulheres:", fnr_women)

FNR Homens: 0.47959183673469385
FNR Mulheres: 0.4666666666666667


### 5.3.2. TabPFN

In [58]:
fnr_men = fnr_by_group(df_test_pfn, 1)
fnr_women = fnr_by_group(df_test_pfn, 0)

print("FNR Homens:", fnr_men)
print("FNR Mulheres:", fnr_women)

FNR Homens: 0.5510204081632653
FNR Mulheres: 0.8666666666666667
